# QuantJourney SDK - Investment Evidence Packet

This notebook demonstrates a QuantJourney SDK workflow that assembles pricing, fundamentals, filings, insiders, identity, short-interest and volatility context for one investable name.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbol = 'AAPL'
benchmark = 'SPY'
prices, volumes = price_panel([symbol, benchmark], start='2022-01-01', end=END)
ratios_raw = qj.fmp.get_financial_ratios_ttm(symbol=symbol)
filings_raw = qj.sec.get_company_filings(symbol=symbol, limit=20)
insiders_raw = qj.sec.get_insider_transactions(symbol=symbol, limit=100)
figi_raw = qj.openfigi.get_figi_data(symbol=symbol, exchange='US')
short_raw = qj.finra.get_short_interest(symbol=symbol)
vix_raw = qj.cboe.get_vix_data(start_date='2022-01-01', end_date=END)


In [ ]:
def first_dict(payload: Any) -> dict[str, Any]:
    value = unwrap(payload)
    if isinstance(value, list) and value:
        return value[0] if isinstance(value[0], dict) else {}
    return value if isinstance(value, dict) else {}

def pick_number(row: dict[str, Any], keys: list[str]) -> float:
    for key in keys:
        if key in row:
            return pd.to_numeric(row.get(key), errors='coerce')
    return np.nan
ratios = first_dict(ratios_raw)
figi = first_dict(figi_raw)
filings = pd.DataFrame(as_rows(filings_raw))
insiders = pd.DataFrame(as_rows(insiders_raw))
short_interest = pd.DataFrame(as_rows(short_raw))
ret = returns(prices)


In [ ]:
evidence = pd.Series({'latest_price': prices[symbol].dropna().iloc[-1], 'return_126d': prices[symbol].pct_change(126).iloc[-1], 'volatility_63d': ret[symbol].tail(63).std() * np.sqrt(252), 'beta_to_spy_252d': ret[[symbol, benchmark]].tail(252).cov().loc[symbol, benchmark] / ret[benchmark].tail(252).var(), 'pe_ttm': pick_number(ratios, ['peRatioTTM', 'pe_ttm', 'priceEarningsRatioTTM']), 'gross_margin_ttm': pick_number(ratios, ['grossProfitMarginTTM', 'gross_margin_ttm']), 'recent_filings': len(filings), 'insider_events': len(insiders), 'short_interest_rows': len(short_interest)})
identity = pd.Series({'composite_figi': figi.get('composite_figi') or figi.get('compositeFIGI') or figi.get('figi'), 'share_class_figi': figi.get('share_class_figi') or figi.get('shareClassFIGI'), 'security_type': figi.get('security_type') or figi.get('securityType'), 'currency': figi.get('currency')}).dropna()
display(evidence)
display(identity)
prices[[symbol, benchmark]].div(prices[[symbol, benchmark]].iloc[0]).plot(title='Evidence packet price context')
plt.ylabel('normalized value')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.